# Story Generation End-to-End Test Demo

This notebook demonstrates the complete CoCoBun `story_generation_v1` workflow:
- AI story text generation (Vertex AI / Gemini)
- Illustration generation for each scene (ComfyUI + Flux)
- Uploading images to GCS

### Prerequisites

- tasks service is running (default `34.44.73.189:8002`)
- tasks-worker is running, and GCP credentials are configured
- ComfyUI is reachable (private network IP)
- Vertex AI API is enabled


## 0. Configuration

In [23]:
# ====== 按需修改 ======
TASKS_URL = "http://34.44.73.189:8002"
SECRET_KEY = "f7a3b9d1e4c6a8f2b5d7e9c1a3f5b7d9e1c3a5f7b9d1e3c5a7f9b1d3e5c7a9f1"  # 和生产 .env 一致
POLL_TIMEOUT = 600  # 故事生成较慢，给 10 分钟
POLL_INTERVAL = 5   # 每 5 秒轮询一次

## 1. Utility Functions

In [24]:
import base64
import hashlib
import hmac
import json
import time
import uuid
import urllib.request
import urllib.error
from datetime import datetime, timedelta, timezone
from IPython.display import display, Image, HTML, Markdown


def b64url(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")


def create_jwt(secret_key: str, user_id: str) -> str:
    header = {"alg": "HS256", "typ": "JWT"}
    payload = {
        "sub": user_id,
        "exp": int((datetime.now(timezone.utc) + timedelta(hours=1)).timestamp()),
        "type": "access",
    }
    signing_input = (
        f"{b64url(json.dumps(header, separators=(',', ':')).encode())}"
        f".{b64url(json.dumps(payload, separators=(',', ':')).encode())}"
    )
    sig = hmac.new(secret_key.encode(), signing_input.encode(), hashlib.sha256).digest()
    return f"{signing_input}.{b64url(sig)}"


def http(method: str, url: str, *, body=None, headers=None):
    req_headers = {"Content-Type": "application/json"}
    if headers:
        req_headers.update(headers)
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, headers=req_headers, method=method)
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            text = resp.read().decode()
            return resp.status, json.loads(text) if text else {}
    except urllib.error.HTTPError as exc:
        text = exc.read().decode("utf-8", errors="replace")
        return exc.code, json.loads(text) if text else {}


def poll_task(tasks_url, auth, task_id, timeout, interval=5):
    """轮询任务直到完成或失败。"""
    deadline = time.time() + timeout
    while time.time() < deadline:
        _, resp = http("GET", f"{tasks_url}/api/v1/tasks/{task_id}", headers=auth)
        d = resp.get("data", resp)
        status = d.get("status")
        stage = d.get("current_stage", "")
        progress = d.get("stage_progress", "")
        print(f"  ⏳ status={status}  stage={stage}  {progress}")
        if status == "completed":
            _, out = http("GET", f"{tasks_url}/api/v1/tasks/{task_id}/outputs", headers=auth)
            return out.get("data", out)
        if status == "failed":
            raise AssertionError(f"Task failed: {d.get('error_message')}")
        time.sleep(interval)
    raise TimeoutError(f"Task {task_id} did not complete within {timeout}s")


# 创建测试用户
user_id = str(uuid.uuid4())
AUTH = {"Authorization": f"Bearer {create_jwt(SECRET_KEY, user_id)}"}
print(f"✅ 测试用户: {user_id}")

✅ Test user: 5fabc4f0-bbe7-4c58-9631-a8df14bc234c


## 2. Check Service Status

In [25]:
status, health = http("GET", f"{TASKS_URL}/health")
print(f"Health: {health}")
assert status == 200, f"Service not healthy: {status}"

_, resp = http("GET", f"{TASKS_URL}/api/v1/tasks/workflows", headers=AUTH)
workflows = resp.get("data", resp).get("workflows", resp.get("workflows", []))
names = [w["name"] for w in workflows]
print(f"\n已注册工作流: {names}")
assert "story_generation_v1" in names, "story_generation_v1 未注册!"
print("\n✅ story_generation_v1 已注册")

Health: {'status': 'ok'}

Registered workflows: ['ping', 'batch_process', 'story_gen_example_1', 'story_gen_example_2', 'story_generation_v1', 'tts_preset_v1', 'tts_clone_v1']

✅ story_generation_v1 is registered


## 3. Create Story Generation Task

In [26]:
create_body = {
    "workflow_name": "story_generation_v1",
    "inputs": {
        "user_id": user_id,
        "age": 5,
        "language": "en",
        "story_type": "daily_life_bedtime",
        "character": "puppy",
        "style": "coco_style",
        "gender_preference": "neutral",
        "specific_topic": "sharing toys kindly",
    },
}

status, resp = http("POST", f"{TASKS_URL}/api/v1/tasks", body=create_body, headers=AUTH)
assert status == 200, f"Create failed: {status} {resp}"

task_data = resp.get("data", resp)
task_id = task_data["id"]

print(f"Task ID:  {task_id}")
print(f"Status:   {task_data['status']}")
print(f"预计耗时: {task_data.get('estimated_seconds', '?')}s")
print(json.dumps(task_data, indent=2, default=str))

Task ID:  ce7d4078-1348-4886-9043-5320a94c5e2c
Status:   pending
Estimated time: 30s
{
  "id": "ce7d4078-1348-4886-9043-5320a94c5e2c",
  "workflow_name": "story_generation_v1",
  "status": "pending",
  "created_at": "2026-04-11T07:43:25.318628Z",
  "estimated_seconds": 30,
  "poll_interval_ms": 2000
}


## 4. Wait for Task Completion

Story generation includes multiple stages: story text generation -> image generation (for each scene) -> GCS upload. It usually takes 2-5 minutes.


In [27]:
result = poll_task(TASKS_URL, AUTH, task_id, POLL_TIMEOUT, POLL_INTERVAL)
print("\n✅ 任务完成!")
print(f"最终状态: {result.get('status', 'completed')}")

  ⏳ status=running  stage=generating_story  None
  ⏳ status=running  stage=generating_story  None
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  stage=generating_images  {'generating_images': {'total': 5, 'completed': 0, 'failed': 0}}
  ⏳ status=running  st

## 5. View Story Content

In [32]:
outputs = result.get("outputs", result)
story = outputs.get("story", {})
scenes = story.get("image_generation", {}).get("scenes", [])
images = outputs.get("images", {}).get("images", [])

print(f"📖 标题: {story.get('title', 'N/A')}")
print(f"📝 摘要: {story.get('summary', 'N/A')}")
print(f"🎭 场景数: {len(scenes)}")
print(f"🖼️ 图片数: {len(images)}")
print(f"📊 字数: {story.get('word_count', 'N/A')}")

📖 Title: Patches Learns to Share
📝 Summary: Patches the puppy learns the joy of sharing their toys with a new friend, Dot. Initially hesitant, Patches discovers that playing together is much more fun than playing alone, leading to a happy afternoon and a valuable lesson about kindness.
🎭 Scene count: 5
🖼️ Image count: 5
📊 Word count: 317


## 6. Display Story Page by Page

In [33]:
# 把 images 按 frame 索引方便查找
image_map = {img.get("frame"): img.get("image_url") for img in images if img.get("image_url")}

for i, scene in enumerate(scenes):
    frame = scene.get("frame", i + 1)
    text = scene.get("story_text", "")
    img_url = image_map.get(frame)
    
    display(Markdown(f"---\n### 第 {frame} 页"))
    display(Markdown(text))
    
    if img_url and img_url.startswith("http"):
        display(Image(url=img_url, width=512))
    elif img_url:
        print(f"  🖼️ {img_url}")
    else:
        print(f"  ⚠️ 该场景无图片")

---
### 第 1 页

Patches the puppy loved their toys. They had a bright red ball, a squeaky blue bone, and a fluffy yellow duck.

---
### 第 2 页

A new puppy named Dot came to visit. Patches watched Dot nervously.

---
### 第 3 页

Mom noticed Patches' hesitation and encouraged them to share.

---
### 第 4 页

Patches and Dot played together, taking turns with the toys.

---
### 第 5 页

Patches realized that sharing wasn't so bad after all.

## 7. Validate Output Integrity

In [34]:
errors = []

if not story.get("title"):
    errors.append("缺少故事标题")

if len(scenes) == 0:
    errors.append("没有场景数据")

for img in images:
    if img.get("error"):
        errors.append(f"场景 {img.get('frame')} 图片生成失败: {img['error']}")
    if not img.get("image_url"):
        errors.append(f"场景 {img.get('frame')} 缺少 image_url (GCS 上传失败?)")

if errors:
    print("❌ 发现问题:")
    for e in errors:
        print(f"  - {e}")
else:
    print("✅ 所有验证通过!")
    print(f"  📖 故事: {story.get('title')}")
    print(f"  🎭 {len(scenes)} 个场景")
    print(f"  🖼️ {len([i for i in images if i.get('image_url')])} 张图片已上传")

print("\n🎉 Story Generation E2E 测试完成!")

✅ All validations passed!
  📖 Story: Patches Learns to Share
  🎭 5  scenes
  🖼️ 5  images uploaded

🎉 Story Generation E2E test completed!


## 8. Raw Output (for debugging)

In [35]:
print(json.dumps(outputs, indent=2, default=str))

{
  "task_id": "ce7d4078-1348-4886-9043-5320a94c5e2c",
  "user_id": "5fabc4f0-bbe7-4c58-9631-a8df14bc234c",
  "status": "completed",
  "seed": 1525563186,
  "created_at": "2026-04-11T07:45:46.228916+00:00",
  "story": {
    "title": "Patches Learns to Share",
    "story": "Patches the puppy loved their toys. They had a bright red ball, a squeaky blue bone, and a fluffy yellow duck. Patches wore a little green vest every day, and today was no different. One sunny afternoon, a new puppy named Dot came to visit. Dot had big, floppy ears and a wagging tail.\n\nPatches watched Dot nervously as Dot sniffed at the red ball. Dot looked up at Patches with hopeful eyes. Patches clutched the ball tightly. \"That's *my* ball,\" Patches mumbled, their tail drooping a little. \n\nMom noticed Patches' hesitation. She knelt down and gently stroked Patches' fur. \"Sharing can be fun, Patches,\" she said softly. \"Think about how happy it would make Dot to play with your ball.\"\n\nPatches thought about